# CDSE Tile Debug\nStep through the exact same request the DAG makes and inspect what Copernicus returns.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

# Load creds from .env
from dotenv import load_dotenv
load_dotenv("../.env")

CDSE_CLIENT_ID     = os.environ["CDSE_CLIENT_ID"]
CDSE_CLIENT_SECRET = os.environ["CDSE_CLIENT_SECRET"]
print("client_id:", CDSE_CLIENT_ID[:12], "...")

In [ ]:
from sentinelhub import (
    BBox, CRS, DataCollection, MimeType,
    SHConfig, SentinelHubRequest, bbox_to_dimensions,
)
import numpy as np

config = SHConfig(
    sh_client_id=CDSE_CLIENT_ID,
    sh_client_secret=CDSE_CLIENT_SECRET,
    sh_base_url="https://sh.dataspace.copernicus.eu",
    sh_token_url="https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
    download_timeout_seconds=120,
)

S2_L2A = DataCollection.SENTINEL2_L2A.define_from(
    "s2l2a_cdse", service_url="https://sh.dataspace.copernicus.eu"
)

print("Config ok:", config.sh_base_url)

## 1 — Tile r0,c0 geometry\nSouthwest corner of the Chicagoland AOI (Gary, IN area). This is the first tile the DAG fetches.

In [ ]:
AOI_BBOX = (-88.4, 41.45, -87.5, 42.75)
lon_min, lat_min, lon_max, lat_max = AOI_BBOX
import math

full_bbox = BBox(bbox=AOI_BBOX, crs=CRS.WGS84)
full_w, full_h = bbox_to_dimensions(full_bbox, resolution=10)
n_cols = math.ceil(full_w / 2500)
n_rows = math.ceil(full_h / 2500)
col_step = (lon_max - lon_min) / n_cols
row_step = (lat_max - lat_min) / n_rows

r, c = 0, 0
tile_bbox = BBox(
    bbox=(lon_min + c*col_step, lat_min + r*row_step,
          lon_min + (c+1)*col_step, lat_min + (r+1)*row_step),
    crs=CRS.WGS84,
)
tile_w, tile_h = bbox_to_dimensions(tile_bbox, resolution=10)

print(f"Full AOI: {full_w}x{full_h}px  →  {n_cols} cols x {n_rows} rows")
print(f"Tile r{r}c{c} bbox: {tile_bbox.lower_left} → {tile_bbox.upper_right}")
print(f"Tile pixel size: {tile_w}x{tile_h}")

## 2 — Raw request (same evalscript as the DAG)\nForce redownload so we bypass any local sentinelhub cache.

In [ ]:
EVALSCRIPT = """
//VERSION=3
function setup() {
  return {
    input: [{ bands: ["B02", "B03", "B04"] }],
    output: { bands: 3, sampleType: "UINT8" },
    mosaicking: "SIMPLE"
  };
}
function evaluatePixel(samples) {
  if (!samples || samples.length === 0) return [0, 0, 0];
  var s = samples[0];
  if (!s) return [0, 0, 0];
  var f = 255.0 / 3000.0;
  return [
    Math.min(255, Math.round(s.B04 * f)),
    Math.min(255, Math.round(s.B03 * f)),
    Math.min(255, Math.round(s.B02 * f)),
  ];
}
"""

req = SentinelHubRequest(
    evalscript=EVALSCRIPT,
    input_data=[SentinelHubRequest.input_data(
        data_collection=S2_L2A,
        time_interval=("2019-06-01", "2019-09-30"),
    )],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=tile_bbox,
    size=(tile_w, tile_h),
    config=config,
)

data = req.get_data(redownload=True)
arr = data[0] if (data and data[0] is not None) else None

if arr is not None:
    print(f"Shape : {arr.shape}  dtype={arr.dtype}")
    print(f"Range : min={arr.min()}  max={arr.max()}  mean={arr.mean():.2f}")
    print(f"Nonzero pixels: {np.count_nonzero(arr.sum(axis=2))} / {arr.shape[0]*arr.shape[1]}")
else:
    print("No data returned")

## 3 — Visualise\nIf we got real pixels, show them. If it's all zeros we'll know for certain the issue is upstream in CDSE.

In [ ]:
import matplotlib.pyplot as plt

if arr is not None and arr.max() > 0:
    plt.figure(figsize=(10, 8))
    plt.imshow(arr)
    plt.title(f"Tile r{r}c{c}  2019-06 to 2019-09  shape={arr.shape}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("All zeros — nothing to show. Check cells below for diagnostics.")

## 4 — Diagnostics (if zeros)\nTry a smaller area, a more recent year, and raw float reflectance to rule out each possible cause.

In [ ]:
# 4a — small bbox, recent date, raw FLOAT32 to bypass the uint8 scaling
EVALSCRIPT_FLOAT = """
//VERSION=3
function setup() {
  return {
    input: [{ bands: ["B04", "B03", "B02"] }],
    output: { bands: 3, sampleType: "FLOAT32" },
    mosaicking: "SIMPLE"
  };
}
function evaluatePixel(samples) {
  return [samples[0].B04, samples[0].B03, samples[0].B02];
}
"""

small_bbox = BBox(bbox=(-87.75, 41.85, -87.65, 41.95), crs=CRS.WGS84)  # downtown Chicago
sw, sh = bbox_to_dimensions(small_bbox, resolution=60)  # coarse res for speed
print(f"Small test bbox pixel size: {sw}x{sh}")

req2 = SentinelHubRequest(
    evalscript=EVALSCRIPT_FLOAT,
    input_data=[SentinelHubRequest.input_data(
        data_collection=S2_L2A,
        time_interval=("2023-07-01", "2023-08-31"),  # recent, definitely has data
    )],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=small_bbox,
    size=(sw, sh),
    config=config,
)

data2 = req2.get_data(redownload=True)
arr2 = data2[0] if (data2 and data2[0] is not None) else None

if arr2 is not None:
    print(f"Shape: {arr2.shape}  dtype={arr2.dtype}")
    print(f"Range: min={arr2.min():.4f}  max={arr2.max():.4f}  mean={arr2.mean():.4f}")
    print(f"Nonzero: {np.count_nonzero(arr2.sum(axis=2))} / {arr2.shape[0]*arr2.shape[1]}")
else:
    print("No data")

In [ ]:
# 4b — visualise the float result if we got data
if arr2 is not None and arr2.max() > 0:
    rgb_vis = np.clip(arr2 / 0.3, 0, 1)  # stretch to visible range
    plt.figure(figsize=(6, 6))
    plt.imshow(rgb_vis)
    plt.title("Downtown Chicago 2023 — raw reflectance")
    plt.axis("off")
    plt.show()
else:
    print("Still zeros — credentials or subscription issue, not a code bug.")

## 5 — Narrow down the cause\nTest the same tile r0c0 bbox with three variables changed one at a time.

In [ ]:
def quick_request(bbox, time_interval, mosaicking="SIMPLE", sample_type="UINT8", resolution=60):
    """Minimal request — 60m resolution for speed."""
    w, h = bbox_to_dimensions(bbox, resolution=resolution)
    scale = f"255.0/3000.0" if sample_type == "UINT8" else "1.0"
    es = f"""
//VERSION=3
function setup() {{
  return {{ input: [{{ bands: ["B04","B03","B02"] }}], output: {{ bands: 3, sampleType: "{sample_type}" }}, mosaicking: "{mosaicking}" }};
}}
function evaluatePixel(samples) {{
  if (!samples || samples.length === 0) return [0,0,0];
  var s = samples[0]; if (!s) return [0,0,0];
  var f = {scale};
  return [Math.min(255,s.B04*f), Math.min(255,s.B03*f), Math.min(255,s.B02*f)];
}}
"""
    r = SentinelHubRequest(
        evalscript=es,
        input_data=[SentinelHubRequest.input_data(data_collection=S2_L2A, time_interval=time_interval)],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=bbox, size=(w, h), config=config,
    )
    d = r.get_data(redownload=True)
    a = d[0] if (d and d[0] is not None) else None
    if a is None:
        return None, "no response"
    nz = int(np.count_nonzero(a.sum(axis=-1)))
    return a, f"shape={a.shape} min={a.min():.3f} max={a.max():.3f} nonzero={nz}/{a.shape[0]*a.shape[1]}"

results = {}

# A: same tile r0c0, same year, different mosaicking
a, msg = quick_request(tile_bbox, ("2019-06-01","2019-09-30"), mosaicking="ORBIT")
results["A: r0c0 2019 ORBIT"]   = msg

# B: same tile r0c0, recent year (should always have data)
b, msg = quick_request(tile_bbox, ("2023-07-01","2023-08-31"), mosaicking="SIMPLE")
results["B: r0c0 2023 SIMPLE"]  = msg

# C: same tile r0c0, recent year, ORBIT
c, msg = quick_request(tile_bbox, ("2023-07-01","2023-08-31"), mosaicking="ORBIT")
results["C: r0c0 2023 ORBIT"]   = msg

# D: different tile — downtown Chicago, 2019
downtown = BBox(bbox=(-87.75, 41.85, -87.65, 41.95), crs=CRS.WGS84)
d, msg = quick_request(downtown,  ("2019-06-01","2019-09-30"), mosaicking="SIMPLE")
results["D: downtown 2019 SIMPLE"] = msg

for k, v in results.items():
    print(f"{k:35s}  →  {v}")

## 6 — Catalog search\nAsk the CDSE catalog directly: do any Sentinel-2 L2A scenes exist for this bbox/time?\nIf this returns 0 items for downtown Chicago 2023 the DataCollection definition is wrong.

In [ ]:
from sentinelhub import SentinelHubCatalog

catalog = SentinelHubCatalog(config=config)

for label, bbox, time in [
    ("downtown 2023", downtown, ("2023-07-01", "2023-08-31")),
    ("tile r0c0  2019", tile_bbox, ("2019-06-01", "2019-09-30")),
]:
    results = list(catalog.search(
        collection=S2_L2A,
        bbox=bbox,
        time=time,
        fields={"include": ["id", "properties.datetime", "properties.eo:cloud_cover"]},
    ))
    print(f"{label}: {len(results)} scene(s) in catalog")
    for r in results[:3]:
        print(f"  {r['id']}  cc={r['properties'].get('eo:cloud_cover','?'):.0f}%")

## 7 — Constant evalscript sanity check\nNo input bands at all. If this returns a solid red image the process API works fine.\nIf it returns zeros, something structural is broken in the request pipeline.

In [ ]:
EVALSCRIPT_CONST = """
//VERSION=3
function setup() {
  return { input: [], output: { bands: 3, sampleType: "UINT8" } };
}
function evaluatePixel(samples) {
  return [255, 0, 0]; // solid red — no satellite data needed
}
"""

w, h = bbox_to_dimensions(downtown, resolution=60)
req_const = SentinelHubRequest(
    evalscript=EVALSCRIPT_CONST,
    input_data=[],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=downtown, size=(w, h), config=config,
)
data_c = req_const.get_data(redownload=True)
arr_c  = data_c[0] if (data_c and data_c[0] is not None) else None

if arr_c is not None:
    print(f"Shape: {arr_c.shape}  R={arr_c[...,0].mean():.0f}  G={arr_c[...,1].mean():.0f}  B={arr_c[...,2].mean():.0f}")
    if arr_c[..., 0].mean() > 200:
        print("✓ Process API is healthy — solid red returned as expected")
    else:
        print("✗ Expected red but got something else — structural issue")
else:
    print("No response at all")